In [ ]:
from collections import defaultdict

import numpy as np
import pandas as pd

In [ ]:
groups = {
    r'$L_2$': ['l2'],
    'ImageNet': ['alex', 'vgg', 'vit_imagenet', 'swin_imagenet', 'clip'],
    'Radiology': ['radimagenet', 'biomedclip'],
    'Ultrasound': ['usfm', 'tusa_vit', 'ultrasound_clip']
}

In [ ]:
metrics = {}

for organ in ['follicles', 'prostate']:
    df = pd.read_csv(f'../assets/results/04/{organ}/metrics.csv')
    df = df.rename(columns={
        **{
        'ssim': r'$\uparrow$ SSIM',
        'hfen': r'$\downarrow$ HFEN'
        }, 
        **{
            g: rf'$\downarrow$ {g.replace('_', ' ')}' for g in df.columns if 'GLCM' in g
        }})
    
    metric_columns = [
        c for c in df.columns
        if c not in ['data_file', 'model']
    ]

    df.model = df.model.apply(
        lambda x: [g for g, l in groups.items() if x in l][0]
    )

    # store means/stds separately
    means = {}
    stds = {}

    for model, subdf in df.groupby('model'):
        means[model] = {
            mc: subdf[mc].mean()
            for mc in metric_columns
        }

        stds[model] = {
            mc: subdf[mc].std()
            for mc in metric_columns
        }

    means_df = pd.DataFrame(means)
    stds_df = pd.DataFrame(stds)
    formatted = pd.DataFrame(
        index=means_df.index,
        columns=means_df.columns
    )
    for row in means_df.index:
        vals = means_df.loc[row]

        # SSIM: maximize
        if 'SSIM' in row:
            best_val = vals.max()

        # everything else: minimize
        else:
            best_val = vals.min()

        # allow ties
        best_cols = vals[vals.round(2) == round(best_val, 2)].index
        formatted_row = {}

        for col in means_df.columns:
            s = f'{means_df.loc[row, col]:.2f} $\\pm$ {stds_df.loc[row, col]:.2f}'

            if col in best_cols:
                s = r'\textbf{' + s + '}'

            formatted_row[col] = s
        formatted.loc[row] = formatted_row
    metrics[organ] = formatted

metrics = {
    'USOVA': metrics['follicles'],
    'MicroSegNet': metrics['prostate']
}

In [ ]:
df2 = pd.concat(metrics, axis=1)
n = len(groups)

latex = df2.to_latex(
    label="tab:inr",
    escape=False,
    index=True,
    column_format=f"l|{'c'*n}|{'c'*n}",
    multicolumn_format="c",
    caption="Image reconstruction metrics of INRs trained with each group of distance metrics"
)

lines = latex.replace(r'\begin{tabular}', r'\centering\resizebox{\textwidth}{!}{\begin{tabular}').replace(r'\end{tabular}', r'\end{tabular}}').split('\n')
print('\n'.join(lines))

In [ ]:
ssims = {}
hfens = {}
for organ, organ_result in inr_results.items():
    ssims[organ] = {t: [] for t in titles.values()}
    hfens[organ] = {t: [] for t in titles.values()}
    for res in organ_result.values():
        ims = res['original']
        for metric in ['l2', 'lpips', 'ultrapips']:
            if metric in res:
                ssims[organ][titles[metric]].append(
                    structural_similarity(ims, res[metric], win_size=5, data_range=2.)
                )
                hfens[organ][titles[metric]].append(
                    hfen(ims, res[metric], sigma=2.5)
                )
            else:
                ssims[organ][titles[metric]].append(float('nan'))
                hfens[organ][titles[metric]].append(float('nan'))
